In [1]:
import os
import rasterio

In [3]:


input_folder = r"C:\Users\admin\Downloads\00_forest"
output_folder = r"C:\Users\admin\Downloads\00_forest\output"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith((".tif", ".tiff")):
        input_path = os.path.join(input_folder, filename)

        with rasterio.open(input_path) as src:

            # Use the same metadata but override count=1 and compression
            profile = src.profile.copy()
            profile.update(
                count=1,
                compress="DEFLATE",
                tiled=True,
                blockxsize=256,
                blockysize=256
            )

            # Output paths
            out1 = os.path.join(output_folder, filename.replace(".tif", "_B1.tif").replace(".tiff", "_B1.tif"))
            out3 = os.path.join(output_folder, filename.replace(".tif", "_B3.tif").replace(".tiff", "_B3.tif"))

            # Create output rasters
            with rasterio.open(out1, "w", **profile) as dst1, \
                 rasterio.open(out3, "w", **profile) as dst3:

                # Loop through windows for block-by-block processing
                for ji, window in src.block_windows(1):  # read windows from band 1's structure
                    # Read each band window
                    b1 = src.read(1, window=window)
                    b3 = src.read(3, window=window)

                    # Write those windows to outputs
                    dst1.write(b1, 1, window=window)
                    dst3.write(b3, 1, window=window)

        print(f"Processed with windowed I/O: {filename}")


Processed with windowed I/O: pyear_11.tif
Processed with windowed I/O: pyear_12.tif
